In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

import cudf
import pandas as pd
import cupy as cp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [17]:
df_train = pd.read_csv('datasets/Quora Questions Pair Dataset/train_preprocessed.csv')
print(type(df_train))
df_train.sample(3)

<class 'pandas.core.frame.DataFrame'>


,question1,question2,is_duplicate
82538,what is the meaning of flagged as needing impr...,how often do you mark questions as needing imp...,1
181918,why are highly productive people well liked,what is the best used car to buy for lakhs in ...,0
51133,how can you find the length and width of a rec...,how do you find the length of the rectangle,0


In [18]:
def tokenize(text):
    return text.split()

In [19]:
all_tokens = []
for col in ['question1', 'question2']:
    for row in df_train[col]:
        all_tokens.extend(tokenize(row))
print(f"Corpus size: {len(all_tokens)}")

Corpus size: 9016420


In [20]:
#get all the words with the number of times they repeat
counter = Counter(all_tokens)
counter

Counter({'the': 381218,
         'what': 324583,
         'is': 303747,
         'a': 280272,
         'i': 226080,
         'how': 220836,
         'in': 217045,
         'to': 212279,
         'do': 170765,
         'of': 164274,
         'are': 149915,
         'and': 135559,
         'can': 114372,
         'for': 104611,
         'you': 96247,
         'my': 87862,
         'why': 84040,
         'it': 75246,
         'best': 70600,
         'on': 60860,
         'does': 59464,
         'or': 46840,
         'have': 44932,
         'be': 44749,
         'if': 44129,
         'which': 43979,
         'some': 41710,
         'that': 40750,
         'get': 40616,
         'with': 39909,
         'should': 39625,
         'not': 39063,
         'an': 37216,
         'from': 35304,
         'your': 34073,
         'india': 29584,
         'will': 29462,
         'people': 26407,
         'when': 26229,
         'who': 26106,
         'at': 25840,
         'like': 25648,
         'good'

In [21]:
# build vocab of all the words with 5 or more occurences
MIN_COUNT = 4
vocab = ['<pad>','<unk>'] + [w for w,c in counter.items() if c >= MIN_COUNT]
vocab

['<pad>',
 '<unk>',
 'what',
 'is',
 'the',
 'step',
 'by',
 'guide',
 'to',
 'invest',
 'in',
 'share',
 'market',
 'india',
 'story',
 'of',
 'kohinoor',
 'kos',
 'i',
 'door',
 'diamond',
 'how',
 'can',
 'increase',
 'speed',
 'my',
 'internet',
 'connection',
 'while',
 'using',
 'a',
 'van',
 'why',
 'am',
 'mentally',
 'very',
 'lonely',
 'solve',
 'it',
 'which',
 'one',
 'dissolve',
 'water',
 'quickly',
 'sugar',
 'salt',
 'methane',
 'and',
 'carbon',
 'oxide',
 'astrology',
 'capricorn',
 'sun',
 'cap',
 'moon',
 'rising',
 'does',
 'that',
 'say',
 'about',
 'me',
 'should',
 'buy',
 'iago',
 'be',
 'good',
 'geologist',
 'when',
 'do',
 'you',
 'use',
 'instead',
 'motorola',
 'company',
 'hack',
 'charter',
 'dix',
 'method',
 'find',
 'separation',
 'slits',
 'fresnel',
 'baptism',
 'read',
 'youtube',
 'comments',
 'make',
 'physics',
 'easy',
 'learn',
 'was',
 'your',
 'first',
 'sexual',
 'experience',
 'like',
 'are',
 'laws',
 'change',
 'status',
 'from',
 'stude

In [22]:
# generate index of all the words
word2idx = {w:i for i,w in enumerate(vocab)}
word2idx

{'<pad>': 0,
 '<unk>': 1,
 'what': 2,
 'is': 3,
 'the': 4,
 'step': 5,
 'by': 6,
 'guide': 7,
 'to': 8,
 'invest': 9,
 'in': 10,
 'share': 11,
 'market': 12,
 'india': 13,
 'story': 14,
 'of': 15,
 'kohinoor': 16,
 'kos': 17,
 'i': 18,
 'door': 19,
 'diamond': 20,
 'how': 21,
 'can': 22,
 'increase': 23,
 'speed': 24,
 'my': 25,
 'internet': 26,
 'connection': 27,
 'while': 28,
 'using': 29,
 'a': 30,
 'van': 31,
 'why': 32,
 'am': 33,
 'mentally': 34,
 'very': 35,
 'lonely': 36,
 'solve': 37,
 'it': 38,
 'which': 39,
 'one': 40,
 'dissolve': 41,
 'water': 42,
 'quickly': 43,
 'sugar': 44,
 'salt': 45,
 'methane': 46,
 'and': 47,
 'carbon': 48,
 'oxide': 49,
 'astrology': 50,
 'capricorn': 51,
 'sun': 52,
 'cap': 53,
 'moon': 54,
 'rising': 55,
 'does': 56,
 'that': 57,
 'say': 58,
 'about': 59,
 'me': 60,
 'should': 61,
 'buy': 62,
 'iago': 63,
 'be': 64,
 'good': 65,
 'geologist': 66,
 'when': 67,
 'do': 68,
 'you': 69,
 'use': 70,
 'instead': 71,
 'motorola': 72,
 'company': 73,
 'h

In [23]:
VOCAB_SIZE = len(vocab)
VOCAB_SIZE

28549

In [24]:
# tokenize a text --> consider upto first 25 words only
# if there are <25 words, append zeros at the end to make 25 words exactly
def encode(sent, max_words=25):
    # in word2idx.get(w,1) --> if w doesn't exist in vocab, return index 1 <'unk'>
    ids = [word2idx.get(w,1) for w in tokenize(sent)][:max_words]
    # append 0 at the end, 0=<'pad'> to make the length 25 exactly
    ids += [0] * (max_words - len(ids))
    return ids

In [25]:
MAX_WORDS = 30
df_train['ids1'] = df_train['question1'].apply(lambda s: encode(s, MAX_WORDS))
df_train['ids2'] = df_train['question2'].apply(lambda s: encode(s, MAX_WORDS))
df_train.sample(5)

,question1,question2,is_duplicate,ids1,ids2
54016,what are some signs that someone is a java a a,what are some signs someone is a a a girlfriend,0,"[2, 96, 159, 999, 57, 146, 3, 30, 363, 30, 30,...","[2, 96, 159, 999, 146, 3, 30, 30, 30, 229, 0, ..."
8293,can you see who viewed your instagram,can someone see if you have viewed public inst...,1,"[22, 69, 755, 319, 3965, 91, 268, 0, 0, 0, 0, ...","[22, 146, 755, 303, 69, 308, 3965, 2384, 268, ..."
276395,is the university of vermont a good school,how good is the university of vermont,1,"[3, 4, 1353, 15, 4119, 30, 65, 1967, 0, 0, 0, ...","[21, 65, 3, 4, 1353, 15, 4119, 0, 0, 0, 0, 0, ..."
125494,what is a chicken tenderloin,what can one do with the flat end of a beef te...,0,"[2, 3, 30, 1589, 5941, 0, 0, 0, 0, 0, 0, 0, 0,...","[2, 22, 40, 68, 125, 4, 3290, 1145, 15, 30, 80..."
9330,is hard work best or smart work best,which one is the best to choose smart work or ...,1,"[3, 1675, 493, 137, 223, 1001, 493, 137, 0, 0,...","[39, 40, 3, 4, 137, 8, 234, 1001, 493, 223, 16..."


In [26]:
# Skip-gram Word2Vec with negative sampling (using custom nn.Module on GPU)

WINDOW = 4
EMBED_DIM = 128
NEG_SAMPLES = 5

In [27]:
# for each row, a pair is (curr_word, +-WINDOW word)
def build_skipgram_pairs(token_list, window=WINDOW):
    pairs = []
    for tokens in token_list:
        ids = [word2idx.get(w,1) for w in tokens]
        n = len(ids)
        for i, center in enumerate(ids):
            lo = max(0, i - window)
            hi = min(n, i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    pairs.append((center, ids[j]))
    return pairs

In [28]:
sentence_tokens = [tokenize(s) for s in pd.concat([df_train['question1'],df_train['question2']])]
pairs = build_skipgram_pairs(sentence_tokens)
pairs

[(2, 3),
 (2, 4),
 (2, 5),
 (2, 6),
 (3, 2),
 (3, 4),
 (3, 5),
 (3, 6),
 (3, 5),
 (4, 2),
 (4, 3),
 (4, 5),
 (4, 6),
 (4, 5),
 (4, 7),
 (5, 2),
 (5, 3),
 (5, 4),
 (5, 6),
 (5, 5),
 (5, 7),
 (5, 8),
 (6, 2),
 (6, 3),
 (6, 4),
 (6, 5),
 (6, 5),
 (6, 7),
 (6, 8),
 (6, 9),
 (5, 3),
 (5, 4),
 (5, 5),
 (5, 6),
 (5, 7),
 (5, 8),
 (5, 9),
 (5, 10),
 (7, 4),
 (7, 5),
 (7, 6),
 (7, 5),
 (7, 8),
 (7, 9),
 (7, 10),
 (7, 11),
 (8, 5),
 (8, 6),
 (8, 5),
 (8, 7),
 (8, 9),
 (8, 10),
 (8, 11),
 (8, 12),
 (9, 6),
 (9, 5),
 (9, 7),
 (9, 8),
 (9, 10),
 (9, 11),
 (9, 12),
 (9, 10),
 (10, 5),
 (10, 7),
 (10, 8),
 (10, 9),
 (10, 11),
 (10, 12),
 (10, 10),
 (10, 13),
 (11, 7),
 (11, 8),
 (11, 9),
 (11, 10),
 (11, 12),
 (11, 10),
 (11, 13),
 (12, 8),
 (12, 9),
 (12, 10),
 (12, 11),
 (12, 10),
 (12, 13),
 (10, 9),
 (10, 10),
 (10, 11),
 (10, 12),
 (10, 13),
 (13, 10),
 (13, 11),
 (13, 12),
 (13, 10),
 (2, 3),
 (2, 4),
 (2, 14),
 (2, 15),
 (3, 2),
 (3, 4),
 (3, 14),
 (3, 15),
 (3, 16),
 (4, 2),
 (4, 3),
 (4, 14)

In [29]:
pairs = np.array(pairs, dtype=np.int64)
pairs.shape

(55983960, 2)